# LoRA & QLoRA students — big students on small hardware

Full fine-tuning caps laptop students at a few hundred million parameters, because AdamW is a memory hog: for every trainable weight you hold the weight, its gradient, and **two optimizer moments** — roughly 16 bytes/param in fp32. A 1.5B student would want ~24GB before activations.

**LoRA** breaks that wall: the base model is *frozen* (weights only, no gradients, no optimizer state) and small low-rank adapter matrices train on top — typically **<1% of the parameters**. That's how a 1–3B student fits on a 16GB MacBook.

This notebook, in order:

1. [Setup](#1) — install with the `[lora]` extra
2. [A tiny dataset](#2) — teacher-generated, ~1 min
3. [LoRA distillation](#3) — watch the trainable-parameter count collapse
4. [Merged checkpoints](#4) — the output is a plain HF model, LoRA leaves no trace
5. [Adapter-only saves](#5) — checkpoints measured in MB, not GB
6. [QLoRA](#6) — 4-bit frozen base (CUDA only, honestly guarded)
7. [Scaling up](#7) — the 1.5B-on-a-MacBook recipe

Runtime: ~5 min on Apple Silicon after the ~1GB SmolLM2 download. Every cell below was executed on a 16GB MacBook (MPS) — the outputs you see are real.

<a id="1"></a>
## 1. Setup

The `[lora]` extra adds [peft](https://github.com/huggingface/peft). (QLoRA additionally needs `[qlora]` → bitsandbytes, CUDA only.)

In [1]:
%pip install -q "distill-anything[lora]"
# uv users, from a terminal:  uv pip install "distill-anything[lora]"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import distillanything
from distillanything.hardware import device_summary

print("distill-anything:", distillanything.__version__)
print("device:          ", device_summary())

distill-anything: 0.2.0
device:           mps (Apple Silicon)


<a id="2"></a>
## 2. A tiny dataset (teacher-generated)

Small on purpose — this notebook demonstrates the *mechanics*; for a real student you'd seed hundreds of domain prompts and let the teacher expand them (see the [main tour notebook](sample_notebook.ipynb)).

In [3]:
from pathlib import Path

from distillanything.data.generate import generate_dataset
from distillanything.teachers.registry import resolve_teacher

# The teacher is loaded in HALF PRECISION automatically on MPS/CUDA — teachers
# only run inference, so fp16 halves their memory. (That's what lets a 3B
# teacher sit next to a 1.5B student on 16GB — see section 7.)
TEACHER = "hf:HuggingFaceTB/SmolLM2-360M-Instruct"

Path("seeds.txt").write_text(
    "Explain the difference between a list and a tuple in Python.\n"
    "What is an API rate limit?\n"
    "Explain gradient descent using a hiking analogy.\n"
    "What is the difference between concurrency and parallelism?\n"
    "Why do databases use indexes?\n"
    "What does HTTP 429 mean?\n"
)

teacher = resolve_teacher(TEACHER)
records = generate_dataset(
    teacher,
    seeds_path="seeds.txt",
    out_path="data/lora_train.jsonl",
    max_tokens=128,
)

/private/tmp/claude-501/-Users-sonu-Desktop-projects-distillanything/15c97c66-dfb7-494c-b5e9-1d6c03b2aa75/scratchpad/nbrun/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 5669.04it/s]

Loaded 6 seed prompts from seeds.txt

generated 6/6

Wrote 6 records to data/lora_train.jsonl

<a id="3"></a>
## 3. LoRA distillation

One argument turns any student into a LoRA student: `lora={...}`. The knobs:

| Knob | Meaning | Rule of thumb |
|---|---|---|
| `r` | rank of the adapter matrices | 8–16 small models, 32–64 for 3B+ |
| `alpha` | adapter scaling (effective LR multiplier) | 2×`r` is the common default |
| `dropout` | regularization on the adapter path | 0.05 |
| `target_modules` | which layers get adapters | leave unset — peft auto-detects known architectures |

Watch the printout: the 135M student trains **well under 2%** of its parameters. Also note `gradient_checkpointing=True` — activations are recomputed in the backward pass instead of stored, the second half of the laptop memory budget.

In [4]:
from distillanything import Student

student = Student(
    "HuggingFaceTB/SmolLM2-135M-Instruct",
    lora={"r": 16, "alpha": 32},          # <- the whole LoRA opt-in
)

results = student.learn(
    teacher=TEACHER,                        # hf: teacher -> white-box logit KD
    dataset="data/lora_train.jsonl",
    max_steps=6,                            # demo-scale; real runs use epochs
    batch_size=2,
    grad_accum=4,
    lr=2e-4,                                # LoRA trains hotter than full FT
    top_k=128,                              # truncated KD: memory saver
    gradient_checkpointing=True,
    output_dir="runs/lora-student",
)

results.get("eval")

Loading student HuggingFaceTB/SmolLM2-135M-Instruct on mps (Apple Silicon)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 5862.06it/s]

LoRA r=16: 0.9M trainable of 135.4M params (0.68%)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.30it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.29it/s]

Saved student to runs/lora-student

{'eval_loss': 0.9854, 'perplexity': 2.679, 'teacher_agreement': 0.7674}

<a id="4"></a>
## 4. Merged checkpoints — LoRA leaves no trace

By default (`merge_lora: true`) the adapters were folded into the base at save time, so `runs/lora-student` is a **plain Hugging Face checkpoint**: it loads with vanilla `AutoModelForCausalLM`, deploys with vLLM/llama.cpp, and benchmarks identically to a fully fine-tuned model. No peft needed downstream.

In [5]:
from transformers import AutoModelForCausalLM  # note: NOT a peft class

reloaded = AutoModelForCausalLM.from_pretrained("runs/lora-student")
print("loads as a plain HF model:", type(reloaded).__name__)
print("adapter files in the dir? ", any(Path("runs/lora-student").glob("adapter*")))

print(student.generate("Why do databases use indexes?", max_new_tokens=80))

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 4579.84it/s]

loads as a plain HF model: LlamaForCausalLM
adapter files in the dir?  False


Databases use indexes to improve the performance of their query execution plans. An index is a data structure that stores frequently occurring data values, such as dates, times, or strings, in a way that allows the database to quickly locate them.

When a query is executed, the database uses the index to quickly locate the data that is most frequently used. This is especially important when the data is


<a id="5"></a>
## 5. Adapter-only saves — checkpoints in MB, not GB

Set `merge_lora=False` and the checkpoint is *just the adapters* — ideal for keeping many experiment variants, or shipping a domain adapter next to a shared base. Every `distill` command (and `Student(...)`) detects adapter dirs and merges them on load, so the small save costs you nothing in usability.

In [6]:
adapter_student = Student(
    "HuggingFaceTB/SmolLM2-135M-Instruct", lora={"r": 16, "alpha": 32}
)
adapter_student.learn(
    teacher=TEACHER,
    dataset="data/lora_train.jsonl",
    max_steps=2,
    batch_size=2, grad_accum=2,
    merge_lora=False,                       # <- adapters only
    output_dir="runs/lora-adapters",
)

def dir_mb(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e6

print(f"merged checkpoint : {dir_mb('runs/lora-student'):8.1f} MB")
print(f"adapter-only save : {dir_mb('runs/lora-adapters'):8.1f} MB")

# The universal loader merges adapter dirs transparently:
tiny_checkpoint = Student("runs/lora-adapters")
print(tiny_checkpoint.generate("What does HTTP 429 mean?", max_new_tokens=60))

Loading student HuggingFaceTB/SmolLM2-135M-Instruct on mps (Apple Silicon)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 21110.84it/s]

LoRA r=16: 0.9M trainable of 135.4M params (0.68%)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

...

merged checkpoint :    272.6 MB
adapter-only save :      7.2 MB


Loading student runs/lora-adapters on mps (Apple Silicon)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 272/272 [00:00<00:00, 6768.22it/s]

HTTP 429 is a common HTTP error code that indicates that a server is unable to deliver a response. It is often used to indicate that a server is experiencing a server-side error, such as a server timeout, a server that is not responding, or a server that is not responding


<a id="6"></a>
## 6. QLoRA — 4-bit frozen base (CUDA only)

QLoRA loads the *frozen base* in 4-bit NF4 (via bitsandbytes) and trains LoRA adapters on top — a further ~4× cut in base-model memory, which is how a 3B student trains on a single consumer GPU.

**bitsandbytes has no Apple Silicon backend**, so this cell is guarded: on a Mac it prints the explanation instead of pretending. (Distill Anything raises the same clear error if you try `qlora: true` off-CUDA.) On a CUDA machine it runs as-is — same API, one extra flag. QLoRA checkpoints always save adapter-only (merging into a 4-bit base isn't supported); as shown in section 5, everything loads them transparently.

In [7]:
import torch

if torch.cuda.is_available():
    # %pip install -q "distill-anything[qlora]"   # peft + bitsandbytes
    qlora_student = Student(
        "Qwen/Qwen2.5-3B-Instruct",
        lora={"r": 32, "alpha": 64, "qlora": True},   # 4-bit NF4 frozen base
    )
    qlora_student.learn(
        teacher="hf:Qwen/Qwen2.5-7B-Instruct",
        dataset="data/lora_train.jsonl",
        epochs=2, batch_size=2, grad_accum=8,
        gradient_checkpointing=True,
        output_dir="runs/qlora-student",
    )
else:
    print("No CUDA here — QLoRA needs bitsandbytes (no Apple Silicon/CPU backend).")
    print("On a Mac, plain LoRA (sections 3-5) is the right tool.")
    print("On a CUDA box: pip install 'distill-anything[qlora]' and run this cell,")
    print("or use the ready-made recipe:  distill train recipes/gpu-qlora.yaml")

No CUDA here — QLoRA needs bitsandbytes (no Apple Silicon/CPU backend).
On a Mac, plain LoRA (sections 3-5) is the right tool.
On a CUDA box: pip install 'distill-anything[qlora]' and run this cell,
or use the ready-made recipe:  distill train recipes/gpu-qlora.yaml


<a id="7"></a>
## 7. Scaling up: 1.5B on a 16GB MacBook

This notebook used small models so it runs in minutes. The real target is [`recipes/mac-lora.yaml`](../recipes/mac-lora.yaml) — **Qwen2.5-3B (fp16 teacher) distills into Qwen2.5-1.5B + LoRA** on 16GB of RAM. The budget that makes it fit:

| Component | Memory | Why it fits |
|---|---:|---|
| Teacher 3B, fp16, inference-only | ~6 GB | half precision, no gradients |
| Student 1.5B base, frozen | ~6 GB | weights only — no grads, no AdamW state |
| LoRA adapters + optimizer | ~0.1 GB | <1% of params train |
| Activations | ~1–2 GB | gradient checkpointing + batch 1 + `top_k` KD |

```bash
pip install "distill-anything[lora]"
distill generate seeds.txt --teacher hf:Qwen/Qwen2.5-3B-Instruct --out data/train.jsonl --expand 5
distill train recipes/mac-lora.yaml
distill report runs/mac-lora --dataset data/train.jsonl --teacher hf:Qwen/Qwen2.5-3B-Instruct --judge claude
```

More: [main tour notebook](sample_notebook.ipynb) · [example scripts](01_generate_dataset.py) · [tests as living docs](../tests/test_lora.py)